# Project 2 IF5150 Inteligensi Artifisial Lanjut
## Notebook 01: Pipeline Preprocessing

Kelompok 3: Chatelia Dyah Prameswari (23526004), Ishak (23526025), Eka Ardilah Febriyanti (23526038).

Notebook ini memuat tahapan pembersihan data, pemisahan train dan validasi, penanganan outlier, encoding dan scaling, seleksi fitur, serta penanganan data tidak seimbang pada dataset performa akademik mahasiswa (target `Target`: Dropout, Enrolled, Graduate). Seluruh transformer dipelajari hanya dari data train. Kelas yang dipakai berada pada `src/preprocessing.py` dan konfigurasi pada `src/config.py`.

## 1. Setup dan konfigurasi
Impor modul, pemuatan data, dan pemeriksaan konsistensi antara train, test, dan sample submission.

In [1]:
# @title 1. Setup & Konfigurasi
import sys, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import (SEED, VAL_SIZE, TARGET, ID_COL, LABEL_MAP, INV_LABEL,
                        NOMINAL_COLS, BINARY_COLS, ORDINAL_COLS, NUMERIC_COLS, FEATURE_COLS,
                        CLIP_STANDARD, CLIP_ZERO_AWARE, CLIP_SKIPPED, OUTLIER_IQR_MULTIPLIER)
from src.preprocessing import (clean_columns, find_data_dir, load_datasets, audit_data,
                               OutlierClipper, HybridFeatureSelector, make_encoder_scaler,
                               random_oversample_indices, prepare_data, load_preprocessed)

random.seed(SEED); np.random.seed(SEED)

DATA_DIR = find_data_dir()
ART_DIR = cfg.ARTIFACT_DIR
FIG_DIR = cfg.FIGURE_DIR
ART_DIR.mkdir(exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

def section(title, width=70):
    """Mencetak judul bagian agar output terstruktur."""
    print("\n" + "=" * width + f"\n{title}\n" + "=" * width)

train_raw, test_raw, sub_tmpl = load_datasets(DATA_DIR)

print("Folder data :", DATA_DIR.relative_to(ROOT))
print("Shape train :", train_raw.shape)
print("Shape test  :", test_raw.shape)
print("Shape submit:", sub_tmpl.shape)

feat_train = [c for c in train_raw.columns if c != TARGET]
print("\nKolom fitur train == test :", feat_train == list(test_raw.columns))
print("Student_ID unik (train)   :", train_raw[ID_COL].is_unique)
print("Student_ID unik (test)    :", test_raw[ID_COL].is_unique)
print("Irisan ID train-test      :", len(set(train_raw[ID_COL]) & set(test_raw[ID_COL])))
print("ID test == ID submission  :", set(test_raw[ID_COL]) == set(sub_tmpl[ID_COL]))

print("\nTotal missing value (train):", int(train_raw.isna().sum().sum()))
print("Total missing value (test) :", int(test_raw.isna().sum().sum()))
print("Baris duplikat (train, tanpa ID):", int(train_raw.drop(columns=ID_COL).duplicated().sum()))
print("Nilai Target tak dikenal   :", sorted(set(train_raw[TARGET]) - set(LABEL_MAP)))

print("\nSemua kolom terkelompok:", set(FEATURE_COLS) | {ID_COL} == set(feat_train))
for name, cols in [("Nominal", NOMINAL_COLS), ("Binary", BINARY_COLS),
                   ("Ordinal", ORDINAL_COLS), ("Numeric", NUMERIC_COLS)]:
    print(f"{name:8s}: {len(cols):2d} kolom")

Folder data : src
Shape train : (3096, 38)
Shape test  : (1328, 37)
Shape submit: (1328, 2)

Kolom fitur train == test : True
Student_ID unik (train)   : True
Student_ID unik (test)    : True
Irisan ID train-test      : 0
ID test == ID submission  : True

Total missing value (train): 0
Total missing value (test) : 0
Baris duplikat (train, tanpa ID): 0
Nilai Target tak dikenal   : []

Semua kolom terkelompok: True
Nominal :  9 kolom
Binary  :  8 kolom
Ordinal :  1 kolom
Numeric : 18 kolom


## 2. Exploratory Data Analysis
Distribusi target, kardinalitas kolom nominal, proporsi outlier, korelasi antar fitur akademik, dan pemeriksaan nilai tidak valid.

In [2]:
# @title 2. Exploratory Data Analysis (EDA)

# 2.1 Distribusi target
cnt = train_raw[TARGET].value_counts()
pct = (cnt / cnt.sum() * 100).round(2)
print(pd.DataFrame({"jumlah": cnt, "persen": pct}))
print("\nRasio kelas mayoritas : minoritas = %.2f : 1" % (cnt.max() / cnt.min()))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(cnt.index, cnt.values, color=["#4c72b0", "#dd8452", "#55a868"])
for i, v in enumerate(cnt.values):
    ax.text(i, v, f"{v}\n({pct.iloc[i]}%)", ha="center", va="bottom", fontsize=9)
ax.set_title("Distribusi kelas Target"); ax.set_ylim(0, cnt.max() * 1.2)
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_target.png", dpi=120); plt.show()

# 2.2 Kardinalitas kolom nominal dan proyeksi lebar one-hot
card = train_raw[NOMINAL_COLS].nunique().sort_values(ascending=False)
n_onehot = int(card.sum())
n_other  = len(BINARY_COLS) + len(ORDINAL_COLS) + len(NUMERIC_COLS)
print("\nKardinalitas kolom nominal:")
print(card.to_string())
print(f"\nJumlah fitur sebelum one-hot : {len(feat_train) - 1}")
print(f"Jumlah fitur sesudah one-hot : {n_onehot + n_other} (nominal {n_onehot} + lainnya {n_other})")

# 2.3 Statistik, skewness, dan proporsi outlier IQR (kolom numerik, data train)
num = train_raw[NUMERIC_COLS]
q1, q3 = num.quantile(0.25), num.quantile(0.75)
iqr = q3 - q1
out15 = ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).mean() * 100
out30 = ((num < q1 - 3.0 * iqr) | (num > q3 + 3.0 * iqr)).mean() * 100
stat = pd.DataFrame({"min": num.min(), "median": num.median(), "max": num.max(),
                     "skew": num.skew(), "%out_1.5IQR": out15, "%out_3IQR": out30}).round(2)
print("\nRingkasan kolom numerik:")
print(stat.to_string())

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
show = ["Age at enrollment", "Admission grade", "Previous qualification (grade)",
        "Curricular units 1st sem (enrolled)", "Curricular units 1st sem (approved)",
        "Curricular units 1st sem (grade)", "Curricular units 2nd sem (approved)",
        "Curricular units 2nd sem (grade)"]
for ax, c in zip(axes.ravel(), show):
    ax.boxplot(train_raw[c], vert=False, flierprops=dict(markersize=3))
    ax.set_title(c, fontsize=8)
plt.suptitle("Boxplot kolom numerik terpilih (sebelum clipping)")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_boxplot.png", dpi=120); plt.show()

# 2.4 Korelasi antar fitur akademik semester 1 dan 2 (dasar pembuangan redundansi)
acad = [c for c in NUMERIC_COLS if c.startswith("Curricular")]
corr = train_raw[acad].corr()
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
         .stack().rename("corr").reset_index()
         .assign(abs_corr=lambda d: d["corr"].abs()).sort_values("abs_corr", ascending=False))
print("\n10 pasangan fitur akademik dengan korelasi tertinggi:")
print(pairs.head(10)[["level_0", "level_1", "corr"]].round(3).to_string(index=False))
print(f"Pasangan dengan |r| > 0.85: {(pairs['abs_corr'] > 0.85).sum()} dari {len(pairs)}")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(acad))); ax.set_yticks(range(len(acad)))
short = [c.replace("Curricular units ", "CU ") for c in acad]
ax.set_xticklabels(short, rotation=90, fontsize=7); ax.set_yticklabels(short, fontsize=7)
plt.colorbar(im); ax.set_title("Korelasi fitur akademik")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_corr.png", dpi=120); plt.show()

# 2.5 Nilai tidak valid (logika domain)
inv = {
 "approved > enrolled (sem 1)": (train_raw["Curricular units 1st sem (approved)"] > train_raw["Curricular units 1st sem (enrolled)"] + train_raw["Curricular units 1st sem (credited)"]).sum(),
 "approved > enrolled (sem 2)": (train_raw["Curricular units 2nd sem (approved)"] > train_raw["Curricular units 2nd sem (enrolled)"] + train_raw["Curricular units 2nd sem (credited)"]).sum(),
 "nilai sem 1 = 0 padahal approved > 0": ((train_raw["Curricular units 1st sem (grade)"] == 0) & (train_raw["Curricular units 1st sem (approved)"] > 0)).sum(),
 "nilai sem 2 = 0 padahal approved > 0": ((train_raw["Curricular units 2nd sem (grade)"] == 0) & (train_raw["Curricular units 2nd sem (approved)"] > 0)).sum(),
 "usia < 17 atau > 70": ((train_raw["Age at enrollment"] < 17) | (train_raw["Age at enrollment"] > 70)).sum(),
}
print("\nPengecekan nilai tidak valid:")
for k, v in inv.items():
    print(f"  {k:42s}: {int(v)}")

          jumlah  persen
Target                  
Graduate    1546   49.94
Dropout      994   32.11
Enrolled     556   17.96

Rasio kelas mayoritas : minoritas = 2.78 : 1



Kardinalitas kolom nominal:
Father's occupation       40
Father's qualification    32
Mother's occupation       31
Mother's qualification    28
Application mode          18
Course                    17
Nationality               17
Previous qualification    17
Marital status             6

Jumlah fitur sebelum one-hot : 36
Jumlah fitur sesudah one-hot : 233 (nominal 206 + lainnya 27)

Ringkasan kolom numerik:
                                                  min  median     max  skew  %out_1.5IQR  %out_3IQR
Previous qualification (grade)                  95.00  133.10  190.00  0.30         3.75       0.10
Admission grade                                 95.00  126.00  190.00  0.53         1.84       0.10
Age at enrollment                               17.00   20.00   70.00  2.08         9.92       3.39
Curricular units 1st sem (credited)              0.00    0.00   20.00  4.19        13.21      13.21
Curricular units 1st sem (enrolled)              0.00    6.00   26.00  1.59         9.7


10 pasangan fitur akademik dengan korelasi tertinggi:
                               level_0                                level_1  corr
   Curricular units 1st sem (credited)    Curricular units 2nd sem (credited) 0.951
   Curricular units 1st sem (enrolled)    Curricular units 2nd sem (enrolled) 0.946
   Curricular units 1st sem (approved)    Curricular units 2nd sem (approved) 0.904
      Curricular units 1st sem (grade)       Curricular units 2nd sem (grade) 0.838
Curricular units 1st sem (evaluations) Curricular units 2nd sem (evaluations) 0.775
   Curricular units 1st sem (enrolled)    Curricular units 1st sem (approved) 0.770
   Curricular units 1st sem (credited)    Curricular units 1st sem (enrolled) 0.768
   Curricular units 2nd sem (approved)       Curricular units 2nd sem (grade) 0.765
   Curricular units 1st sem (enrolled)    Curricular units 2nd sem (credited) 0.752
   Curricular units 1st sem (approved)    Curricular units 2nd sem (enrolled) 0.739
Pasangan dengan |r| >


Pengecekan nilai tidak valid:
  approved > enrolled (sem 1)               : 0
  approved > enrolled (sem 2)               : 0
  nilai sem 1 = 0 padahal approved > 0      : 0
  nilai sem 2 = 0 padahal approved > 0      : 0
  usia < 17 atau > 70                       : 0


## 3. Data cleaning dan train-validation split
Audit kualitas data, pemisahan fitur dan label, split stratified 80/20, dan pemeriksaan nol struktural pada kolom numerik.

In [3]:
# @title 3. Data Cleaning & Train-Validation Split

from sklearn.model_selection import train_test_split

# ----------------------------------------------------------------------
# 3.1 Audit kualitas data (aturan berbasis logika domain, tanpa statistik terpelajar)
# ----------------------------------------------------------------------
section("3.1 Audit kualitas data")

audit_table = pd.concat([audit_data(train_raw).rename("train"),
                         audit_data(test_raw).rename("test")], axis=1)
print(audit_table.to_string())
print("\nKesimpulan: tidak ada baris yang perlu dibuang atau diimputasi."
      if audit_table.values.sum() == 0 else
      "\nPERHATIAN: ada temuan, periksa tabel di atas sebelum melanjutkan.")

# ----------------------------------------------------------------------
# 3.2 Pemisahan fitur, label, dan identitas
# ----------------------------------------------------------------------
section("3.2 Pemisahan fitur, label, dan identitas")

y_all  = train_raw[TARGET].map(LABEL_MAP).astype(int)
X_all  = train_raw.drop(columns=[ID_COL, TARGET]).reset_index(drop=True)
id_all = train_raw[ID_COL].reset_index(drop=True)
y_all  = y_all.reset_index(drop=True)

X_test_raw = test_raw.drop(columns=[ID_COL]).reset_index(drop=True)
id_test    = test_raw[ID_COL].reset_index(drop=True)

print(f"Fitur train : {X_all.shape}  | Label : {y_all.shape}")
print(f"Fitur test  : {X_test_raw.shape}")
print(f"Pemetaan label: {LABEL_MAP}")

# ----------------------------------------------------------------------
# 3.3 Split stratified (dilakukan SEBELUM seluruh langkah fit apa pun)
# ----------------------------------------------------------------------
section("3.3 Train-validation split (stratified)")

idx_tr, idx_val = train_test_split(
    np.arange(len(X_all)), test_size=VAL_SIZE, stratify=y_all, random_state=SEED)

X_train_raw, X_val_raw = X_all.iloc[idx_tr].reset_index(drop=True), X_all.iloc[idx_val].reset_index(drop=True)
y_train, y_val         = y_all.iloc[idx_tr].reset_index(drop=True), y_all.iloc[idx_val].reset_index(drop=True)
id_train, id_val       = id_all.iloc[idx_tr].reset_index(drop=True), id_all.iloc[idx_val].reset_index(drop=True)

dist = pd.DataFrame({
    "train_n"  : y_train.value_counts().sort_index(),
    "train_%"  : (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "val_n"    : y_val.value_counts().sort_index(),
    "val_%"    : (y_val.value_counts(normalize=True).sort_index() * 100).round(2),
    "full_%"   : (y_all.value_counts(normalize=True).sort_index() * 100).round(2),
})
dist.index = [INV_LABEL[i] for i in dist.index]
print(dist.to_string())
print(f"\nUkuran train: {len(X_train_raw)} | validasi: {len(X_val_raw)} | test (disimpan, tidak disentuh): {len(X_test_raw)}")

# Verifikasi integritas split
assert len(set(id_train) & set(id_val)) == 0, "ID train dan validasi beririsan"
assert len(id_train) + len(id_val) == len(id_all), "Jumlah baris tidak konsisten"
print("Verifikasi: tidak ada irisan ID train-validasi dan seluruh baris terpakai.")

# ID split disimpan sehingga pembagian train dan validasi dapat direproduksi anggota lain
np.savez(f"{ART_DIR}/split_ids.npz", train_ids=id_train.values, val_ids=id_val.values,
         seed=SEED, val_size=VAL_SIZE)
print(f"Split disimpan ke {ART_DIR.relative_to(ROOT)}/split_ids.npz")

# ----------------------------------------------------------------------
# 3.4 Pemeriksaan nol struktural (hanya pada data TRAIN)
# ----------------------------------------------------------------------
section("3.4 Pemeriksaan nol struktural pada kolom numerik (data train)")

u = "Curricular units"
for sem in ["1st", "2nd"]:
    g = X_train_raw[f"{u} {sem} sem (grade)"] == 0
    print(f"Semester {sem}: {g.mean()*100:5.2f}% baris bernilai grade = 0 "
          f"| dari baris tersebut: enrolled = 0 pada {(X_train_raw.loc[g, f'{u} {sem} sem (enrolled)'] == 0).mean()*100:5.1f}%, "
          f"evaluations = 0 pada {(X_train_raw.loc[g, f'{u} {sem} sem (evaluations)'] == 0).mean()*100:5.1f}%, "
          f"approved = 0 pada {(X_train_raw.loc[g, f'{u} {sem} sem (approved)'] == 0).mean()*100:5.1f}%")

rows = []
for c in NUMERIC_COLS:
    s = X_train_raw[c]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    nz = s[s != 0]
    q1n, q3n = nz.quantile([0.25, 0.75]) if len(nz) else (np.nan, np.nan)
    iqr_n = q3n - q1n
    out_all = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100
    out_nz  = ((nz < q1n - 1.5 * iqr_n) | (nz > q3n + 1.5 * iqr_n)).mean() * 100 if len(nz) else np.nan
    rows.append({"kolom": c.replace("Curricular units ", "CU "), "%nol": (s == 0).mean() * 100,
                 "IQR": iqr, "%out_semua": out_all, "%out_tanpa_nol": out_nz})
zero_tbl = pd.DataFrame(rows).set_index("kolom").round(2)
print("\n" + zero_tbl.to_string())

fig, ax = plt.subplots(figsize=(8, 4.2))
plot = zero_tbl[["%out_semua", "%out_tanpa_nol"]].dropna()
pos = np.arange(len(plot)); w = 0.4
ax.barh(pos - w/2, plot["%out_semua"], w, label="Semua baris", color="#dd8452")
ax.barh(pos + w/2, plot["%out_tanpa_nol"], w, label="Tanpa nilai nol", color="#4c72b0")
ax.set_yticks(pos); ax.set_yticklabels(plot.index, fontsize=7); ax.invert_yaxis()
ax.set_xlabel("% baris di luar 1.5 x IQR"); ax.legend(fontsize=8)
ax.set_title("Dampak nol struktural terhadap deteksi outlier IQR")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_zero_structural.png", dpi=120); plt.show()


3.1 Audit kualitas data
                                        train  test
Missing value                               0     0
Baris duplikat (tanpa ID)                   0     0
Approved > enrolled + credited (sem 1)      0     0
Approved > enrolled + credited (sem 2)      0     0
Nilai negatif pada kolom hitungan           0     0
Nilai biner di luar {0, 1}                  0     0
Usia di luar 17-70                          0     0

Kesimpulan: tidak ada baris yang perlu dibuang atau diimputasi.

3.2 Pemisahan fitur, label, dan identitas
Fitur train : (3096, 36)  | Label : (3096,)
Fitur test  : (1328, 36)
Pemetaan label: {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}

3.3 Train-validation split (stratified)
          train_n  train_%  val_n  val_%  full_%
Dropout       795    32.11    199   32.1   32.11
Enrolled      445    17.97    111   17.9   17.96
Graduate     1236    49.92    310   50.0   49.94

Ukuran train: 2476 | validasi: 620 | test (disimpan, tidak disentuh): 1328
Verifika

## 4. Outlier detection dan clipping
Clipping berbasis IQR dengan batas yang dipelajari dari train. Pengali IQR dipilih dengan 5-fold cross-validation pada data train.

In [4]:
# @title 4. Outlier Detection & Clipping (fit hanya pada data train)

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

U = "Curricular units"

# ----------------------------------------------------------------------
# 4.1 Ringkasan keputusan per kolom
# ----------------------------------------------------------------------
section("4.1 Keputusan penanganan outlier per kolom")
print(f"Clip standar     ({len(CLIP_STANDARD)}): {[c.replace(U, 'CU') for c in CLIP_STANDARD]}")
print(f"Clip sadar-nol   ({len(CLIP_ZERO_AWARE)}): {[c.replace(U, 'CU') for c in CLIP_ZERO_AWARE]}")
print(f"Tidak di-clip    ({len(CLIP_SKIPPED)}): {[c.replace(U, 'CU') for c in CLIP_SKIPPED]}")

# ----------------------------------------------------------------------
# 4.2 Jumlah nilai terpotong pada train dan validasi untuk tiap pengali IQR
# ----------------------------------------------------------------------
section("4.2 Jumlah sel terpotong (batas dipelajari dari train)")
K_GRID = [1.5, 2.0, 3.0]
rows = {}
for k in K_GRID:
    oc = OutlierClipper(CLIP_STANDARD, CLIP_ZERO_AWARE, k=k).fit(X_train_raw)
    rows[f"train k={k}"] = oc.count_clipped(X_train_raw)
    rows[f"val k={k}"]   = oc.count_clipped(X_val_raw)
clip_tbl = pd.DataFrame(rows)
clip_tbl.index = [c.replace(U, "CU") for c in clip_tbl.index]
print(clip_tbl.to_string())
print("\nTotal sel terpotong (train):",
      {f"k={k}": int(clip_tbl[f'train k={k}'].sum()) for k in K_GRID},
      f"dari {X_train_raw.shape[0] * len(CLIP_STANDARD + CLIP_ZERO_AWARE)} sel pada kolom yang di-clip")

# ----------------------------------------------------------------------
# 4.3 Pemilihan pengali IQR memakai 5-fold CV pada data TRAIN (clipper di-fit per fold)
# ----------------------------------------------------------------------
section("4.3 Pemilihan pengali IQR (5-fold CV pada train, metrik: Macro F1)")

def make_eval_pipeline(k):
    return Pipeline([
        ("clip", OutlierClipper(CLIP_STANDARD, CLIP_ZERO_AWARE, k=k)),
        ("prep", make_encoder_scaler()),
        ("clf",  LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
res = {}
for name, k in [("tanpa clipping", np.inf)] + [(f"k = {k}", k) for k in K_GRID]:
    sc = cross_val_score(make_eval_pipeline(k), X_train_raw, y_train, cv=cv, scoring="f1_macro")
    res[name] = {"mean_macroF1": sc.mean(), "std": sc.std(), "min": sc.min(), "max": sc.max()}
cv_tbl = pd.DataFrame(res).T.round(4)
print(cv_tbl.to_string())

best = cv_tbl["mean_macroF1"].idxmax()
gap = cv_tbl["mean_macroF1"].max() - cv_tbl.loc["tanpa clipping", "mean_macroF1"]
print(f"\nSkor tertinggi: {best} | selisih terhadap tanpa clipping: {gap:+.4f} "
      f"(simpangan baku antar-fold sekitar {cv_tbl['std'].mean():.4f})")

# ----------------------------------------------------------------------
# 4.4 Terapkan clipper final (fit pada train, transform pada train, validasi, dan test)
# ----------------------------------------------------------------------
section("4.4 Penerapan clipper final")
K_FINAL = OUTLIER_IQR_MULTIPLIER
clipper = OutlierClipper(CLIP_STANDARD, CLIP_ZERO_AWARE, k=K_FINAL).fit(X_train_raw)

X_train_clip = clipper.transform(X_train_raw)
X_val_clip   = clipper.transform(X_val_raw)
X_test_clip  = clipper.transform(X_test_raw)

print(f"k final = {K_FINAL}")
print("Batas yang dipelajari (hanya dari train):")
print(pd.DataFrame(clipper.bounds_, index=["batas_bawah", "batas_atas"]).T
      .rename(index=lambda c: c.replace(U, "CU")).round(2).to_string())
print("\nBentuk data:", X_train_clip.shape, X_val_clip.shape, X_test_clip.shape)

show = ["Age at enrollment", f"{U} 1st sem (enrolled)", f"{U} 1st sem (grade)", f"{U} 2nd sem (grade)"]
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for j, c in enumerate(show):
    axes[0, j].boxplot(X_train_raw[c], vert=False, flierprops=dict(markersize=3))
    axes[0, j].set_title(c.replace(U, "CU"), fontsize=8)
    axes[1, j].boxplot(X_train_clip[c], vert=False, flierprops=dict(markersize=3))
axes[0, 0].set_ylabel("Sebelum", fontsize=9); axes[1, 0].set_ylabel("Sesudah", fontsize=9)
plt.suptitle(f"Sebelum dan sesudah clipping (k = {K_FINAL})")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_clipping.png", dpi=120); plt.show()


4.1 Keputusan penanganan outlier per kolom
Clip standar     (9): ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'CU 1st sem (enrolled)', 'CU 2nd sem (enrolled)', 'CU 1st sem (evaluations)', 'CU 2nd sem (evaluations)', 'CU 1st sem (approved)', 'CU 2nd sem (approved)']
Clip sadar-nol   (2): ['CU 1st sem (grade)', 'CU 2nd sem (grade)']
Tidak di-clip    (7): ['CU 1st sem (credited)', 'CU 1st sem (without evaluations)', 'CU 2nd sem (credited)', 'CU 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP']

4.2 Jumlah sel terpotong (batas dipelajari dari train)
                                train k=1.5  val k=1.5  train k=2.0  val k=2.0  train k=3.0  val k=3.0
Previous qualification (grade)           95         21           16          3            2          1
Admission grade                          46         11           20          5            3          0
Age at enrollment                       251         56          182         39  

                mean_macroF1     std     min     max
tanpa clipping        0.7083  0.0090  0.6949  0.7213
k = 1.5               0.7060  0.0119  0.6849  0.7210
k = 2.0               0.7062  0.0088  0.6924  0.7200
k = 3.0               0.7053  0.0071  0.6999  0.7180

Skor tertinggi: tanpa clipping | selisih terhadap tanpa clipping: +0.0000 (simpangan baku antar-fold sekitar 0.0092)

4.4 Penerapan clipper final
k final = 3.0
Batas yang dipelajari (hanya dari train):
                                batas_bawah  batas_atas
Previous qualification (grade)        80.00      185.00
Admission grade                       66.00      186.58
Age at enrollment                      1.00       43.00
CU 1st sem (enrolled)                 -1.00       13.00
CU 2nd sem (enrolled)                 -1.00       13.00
CU 1st sem (evaluations)              -6.00       22.00
CU 2nd sem (evaluations)              -6.00       22.00
CU 1st sem (approved)                 -6.00       15.00
CU 2nd sem (approved)       

## 5. Feature selection
Seleksi fitur bertahap (dukungan minimum, mutual information, redundansi, pemangkasan) dengan perbandingan metode MI dan RFE.

In [5]:
# @title 5. Feature Selection (Hybrid: Dukungan, Mutual Information, Redundansi, Top-n / RFE)

# ----------------------------------------------------------------------
# 5.1 Pipeline evaluasi: clipper -> encoding/scaling -> seleksi -> model
# ----------------------------------------------------------------------
section("5.1 Perbandingan metode dan jumlah fitur (5-fold CV pada train, metrik: Macro F1)")

def make_pipeline(selector):
    return Pipeline([
        ("clip",   OutlierClipper(CLIP_STANDARD, CLIP_ZERO_AWARE, k=K_FINAL)),
        ("prep",   make_encoder_scaler()),
        ("select", selector),
        ("clf",    LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
    ])

N_GRID = [10, 15, 20, 30, 45, 60, 90]
scenarios = [("tanpa seleksi", "passthrough"),
             ("filter saja (0+A+B)", HybridFeatureSelector(n_features=None, random_state=SEED))]
for n in N_GRID:
    scenarios.append((f"MI  n={n}",  HybridFeatureSelector(n_features=n, method="mi",  random_state=SEED)))
for n in N_GRID:
    scenarios.append((f"RFE n={n}", HybridFeatureSelector(n_features=n, method="rfe", random_state=SEED)))

res = {}
for name, sel in scenarios:
    sc = cross_val_score(make_pipeline(sel), X_train_raw, y_train, cv=cv, scoring="f1_macro")
    res[name] = {"mean_macroF1": sc.mean(), "std": sc.std()}
fs_tbl = pd.DataFrame(res).T.round(4)
print(fs_tbl.to_string())

# Aturan pemilihan (metode MI): jumlah fitur paling kecil dengan skor tidak lebih rendah dari
# (skor tanpa seleksi - simpangan baku antar-fold tanpa seleksi)
base_mean, base_std = fs_tbl.loc["tanpa seleksi", "mean_macroF1"], fs_tbl.loc["tanpa seleksi", "std"]
thr = base_mean - base_std
mi_rows = fs_tbl.loc[[i for i in fs_tbl.index if i.startswith("MI")]]
N_FINAL = min(int(i.split("=")[1]) for i in mi_rows.index if mi_rows.loc[i, "mean_macroF1"] >= thr)
print(f"\nAmbang skor = {base_mean:.4f} - {base_std:.4f} = {thr:.4f}")
print(f"Jumlah fitur paling ringkas (metode MI) yang memenuhi ambang: N_FINAL = {N_FINAL}")
print("Catatan: selisih antar-skenario berada dalam derau antar-fold; pemilihan metode MI "
      "didasarkan pada interpretabilitas dan stabilitas, bukan pada skor yang lebih tinggi.")

# ----------------------------------------------------------------------
# 5.2 Fit final pada train: clip -> prep -> selector
# ----------------------------------------------------------------------
section("5.2 Seleksi final (fit hanya pada train)")

prep = make_encoder_scaler().fit(X_train_clip)
Xtr_enc, Xva_enc, Xte_enc = (prep.transform(d) for d in (X_train_clip, X_val_clip, X_test_clip))
Xtr_enc.columns = Xva_enc.columns = Xte_enc.columns = [c.split("__", 1)[1] for c in Xtr_enc.columns]

selector = HybridFeatureSelector(n_features=N_FINAL, method="mi", random_state=SEED).fit(Xtr_enc, y_train)
print("Jumlah fitur per tahap:", selector.stage_counts_)

X_train_sel = selector.transform(Xtr_enc)
X_val_sel   = selector.transform(Xva_enc)
X_test_sel  = selector.transform(Xte_enc)
print("Bentuk data akhir:", X_train_sel.shape, X_val_sel.shape, X_test_sel.shape)

def source_col(f):
    for c in NOMINAL_COLS:
        if f.startswith(c + "_"):
            return c
    return f
src = pd.Series([source_col(f) for f in selector.selected_]).value_counts()
n_orig = len(NOMINAL_COLS) + len(BINARY_COLS) + len(ORDINAL_COLS) + len(NUMERIC_COLS)
print(f"\nFitur terpilih berasal dari {len(src)} kolom asli (dari {n_orig}):")
print(src.to_string())

print("\nFitur terpilih beserta nilai MI (urut menurun):")
print(selector.mi_[selector.selected_].round(4).rename(index=lambda c: c.replace(U, "CU")).to_string())

# Visualisasi: 25 fitur dengan MI tertinggi dan penanda yang lolos seleksi akhir
top = selector.mi_.head(25)[::-1]
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(range(len(top)), top.values,
        color=["#55a868" if f in selector.selected_ else "#b0b0b0" for f in top.index])
ax.set_yticks(range(len(top))); ax.set_yticklabels([f.replace(U, "CU") for f in top.index], fontsize=7)
ax.set_xlabel("Mutual information terhadap Target")
ax.set_title(f"25 fitur dengan MI tertinggi (hijau = terpilih, n = {N_FINAL})")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_feature_selection.png", dpi=120); plt.show()

import json
json.dump({"method": "mi", "n_final": N_FINAL, "corr_threshold": 0.95, "min_support": 0.01,
           "selected_features": selector.selected_},
          open(f"{ART_DIR}/selected_features.json", "w"), indent=2)
print(f"\nDaftar fitur terpilih disimpan ke {ART_DIR.relative_to(ROOT)}/selected_features.json")


5.1 Perbandingan metode dan jumlah fitur (5-fold CV pada train, metrik: Macro F1)


                     mean_macroF1     std
tanpa seleksi              0.7053  0.0071
filter saja (0+A+B)        0.7063  0.0164
MI  n=10                   0.6945  0.0146
MI  n=15                   0.6950  0.0217
MI  n=20                   0.6995  0.0143
MI  n=30                   0.7064  0.0203
MI  n=45                   0.7030  0.0146
MI  n=60                   0.7002  0.0148
MI  n=90                   0.7077  0.0126
RFE n=10                   0.6956  0.0115
RFE n=15                   0.6997  0.0136
RFE n=20                   0.7043  0.0184
RFE n=30                   0.7045  0.0210
RFE n=45                   0.7107  0.0181
RFE n=60                   0.7014  0.0187
RFE n=90                   0.7076  0.0144

Ambang skor = 0.7053 - 0.0071 = 0.6982
Jumlah fitur paling ringkas (metode MI) yang memenuhi ambang: N_FINAL = 20
Catatan: selisih antar-skenario berada dalam derau antar-fold; pemilihan metode MI didasarkan pada interpretabilitas dan stabilitas, bukan pada skor yang lebih tinggi.

5.

Jumlah fitur per tahap: {'awal': 226, 'setelah 0 (dukungan)': 99, 'setelah A (MI)': 94, 'setelah B (redundansi)': 93, 'akhir (C)': 20}
Bentuk data akhir: (2476, 20) (620, 20) (1328, 20)

Fitur terpilih berasal dari 18 kolom asli (dari 36):
Application mode                          2
Course                                    2
Curricular units 2nd sem (approved)       1
Curricular units 1st sem (approved)       1
Curricular units 2nd sem (grade)          1
Curricular units 1st sem (grade)          1
Curricular units 1st sem (evaluations)    1
Tuition fees up to date                   1
Curricular units 2nd sem (evaluations)    1
Curricular units 2nd sem (enrolled)       1
Scholarship holder                        1
Age at enrollment                         1
Curricular units 1st sem (enrolled)       1
Admission grade                           1
Debtor                                    1
Previous qualification (grade)            1
Gender                                    1
Mother's occ


Daftar fitur terpilih disimpan ke artifacts/selected_features.json


## 6. Oversampling, verifikasi bebas leakage, dan penyimpanan artefak
Perbandingan strategi penanganan data tidak seimbang, pemeriksaan leakage, ringkasan pipeline, dan penyimpanan hasil.

In [6]:
# @title 6. Oversampling (hanya train), Verifikasi Bebas Leakage, dan Penyimpanan Artefak

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, recall_score

# ----------------------------------------------------------------------
# 6.1 Apakah oversampling berguna? Bandingkan di dalam CV (resampling hanya pada fold train)
# ----------------------------------------------------------------------
section("6.1 Perbandingan strategi imbalance (5-fold CV pada train)")

# Data tiap fold dibentuk sekali; seluruh transformer di-fit hanya pada fold-train
folds = []
for tr, va in cv.split(X_train_raw, y_train):
    Xa, Xb = X_train_raw.iloc[tr], X_train_raw.iloc[va]
    ya, yb = y_train.iloc[tr].values, y_train.iloc[va].values
    clp = OutlierClipper(CLIP_STANDARD, CLIP_ZERO_AWARE, k=K_FINAL).fit(Xa)
    pr  = make_encoder_scaler().fit(clp.transform(Xa))
    ea, eb = pr.transform(clp.transform(Xa)), pr.transform(clp.transform(Xb))
    ea.columns = eb.columns = [c.split("__", 1)[1] for c in ea.columns]
    sl = HybridFeatureSelector(n_features=N_FINAL, method="mi", random_state=SEED).fit(ea, ya)
    folds.append((sl.transform(ea).values, ya, sl.transform(eb).values, yb))

def run_scenario(make_model, oversample):
    rows = []
    for Xa, ya, Xb, yb in folds:
        if oversample:
            ix = random_oversample_indices(ya)
            Xa, ya = Xa[ix], ya[ix]
        pred = make_model().fit(Xa, ya).predict(Xb)
        rows.append([f1_score(yb, pred, average="macro"),
                     f1_score(yb, pred, average=None)[LABEL_MAP["Enrolled"]],
                     recall_score(yb, pred, average=None)[LABEL_MAP["Enrolled"]]])
    r = np.array(rows)
    return {"macroF1": r[:, 0].mean(), "std": r[:, 0].std(),
            "F1_Enrolled": r[:, 1].mean(), "Recall_Enrolled": r[:, 2].mean()}

lr_plain = lambda: LogisticRegression(max_iter=3000, random_state=SEED)
lr_bal   = lambda: LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
mlp      = lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=SEED)

res = {
    "LR  tanpa penanganan"        : run_scenario(lr_plain, False),
    "LR  class_weight=balanced"   : run_scenario(lr_bal,   False),
    "LR  oversampling"            : run_scenario(lr_plain, True),
    "MLP tanpa penanganan"        : run_scenario(mlp,      False),
    "MLP oversampling"            : run_scenario(mlp,      True),
}
imb_tbl = pd.DataFrame(res).T.round(4)
print(imb_tbl.to_string())
print("\nCatatan: MLPClassifier tidak mendukung class_weight, sehingga oversampling menjadi satu-satunya "
      "cara yang identik untuk versi from scratch dan scikit-learn.")

# ----------------------------------------------------------------------
# 6.2 Terapkan oversampling final pada TRAIN saja
# ----------------------------------------------------------------------
section("6.2 Oversampling final (hanya data train)")

resample_idx = random_oversample_indices(y_train.values)
X_train_res = X_train_sel.values[resample_idx]
y_train_res = y_train.values[resample_idx]

before = y_train.value_counts().sort_index().rename(index=INV_LABEL)
after  = pd.Series(y_train_res).value_counts().sort_index().rename(index=INV_LABEL)
print(pd.DataFrame({"train_sebelum": before, "train_sesudah": after,
                    "validasi (tidak diubah)": y_val.value_counts().sort_index().rename(index=INV_LABEL)}).to_string())
print(f"\nBentuk train sebelum {X_train_sel.shape} -> sesudah {X_train_res.shape}")

# ----------------------------------------------------------------------
# 6.3 Verifikasi bebas leakage (setiap pernyataan diuji, bukan hanya diklaim)
# ----------------------------------------------------------------------
section("6.3 Verifikasi bebas leakage")

checks = {}
checks["ID train dan validasi tidak beririsan"] = len(set(id_train) & set(id_val)) == 0
checks["ID train dan test tidak beririsan"]     = len(set(id_train) & set(id_test)) == 0
checks["Indeks oversampling hanya merujuk baris train"] = resample_idx.max() < len(X_train_sel)
checks["Validasi tidak berubah (620 baris, tanpa duplikasi)"] = len(X_val_sel) == len(y_val) == 620
# Statistik transformer dibandingkan dengan nilai yang dihitung hanya dari train
nz = X_train_raw["Age at enrollment"]
q1, q3 = nz.quantile([0.25, 0.75])
checks["Batas clipper = dihitung dari train saja"] = np.isclose(clipper.bounds_["Age at enrollment"][1], q3 + K_FINAL * (q3 - q1))
sc = prep.named_transformers_["num"]
num_train_mean = X_train_clip[NUMERIC_COLS + ORDINAL_COLS].mean().values
checks["Mean scaler = mean train (bukan train+validasi)"] = np.allclose(sc.mean_, num_train_mean)
pooled = pd.concat([X_train_clip, X_val_clip])[NUMERIC_COLS + ORDINAL_COLS].mean().values
checks["Mean scaler berbeda dari mean gabungan train+val"] = not np.allclose(sc.mean_, pooled)
checks["Kolom train = validasi = test (urutan sama)"] = (list(X_train_sel.columns) == list(X_val_sel.columns) == list(X_test_sel.columns))
checks["Tidak ada NaN pada data akhir"] = not (X_train_sel.isna().any().any() or X_val_sel.isna().any().any() or X_test_sel.isna().any().any())

for k, v in checks.items():
    print(f"[{'LULUS' if v else 'GAGAL'}] {k}")
assert all(checks.values()), "Ada pemeriksaan leakage yang gagal, hentikan dan periksa."

# ----------------------------------------------------------------------
# 6.4 Ringkasan pipeline
# ----------------------------------------------------------------------
section("6.4 Ringkasan pipeline preprocessing")
summary = pd.DataFrame([
    ["1", "Pembersihan nama kolom, audit data",      "tanpa statistik terpelajar", "train, test",            "(3096, 36)"],
    ["2", "Split stratified 80/20",                  "seed 42",                    "train -> train, validasi", f"{X_train_raw.shape}, {X_val_raw.shape}"],
    ["3", "Outlier clipping (IQR, k=3)",             "train",                      "train, val, test",        "bentuk tetap"],
    ["4", "Encoding + scaling",        "train",                      "train, val, test",        f"{Xtr_enc.shape[1]} fitur"],
    ["5", "Feature selection (MI, n=%d)" % N_FINAL,  "train",                      "train, val, test",        f"{X_train_sel.shape[1]} fitur"],
    ["6", "Random oversampling",                     "train",                      "train saja",              f"{X_train_res.shape}"],
], columns=["No", "Langkah", "Di-fit pada", "Diterapkan ke", "Keluaran"])
print(summary.to_string(index=False))

# ----------------------------------------------------------------------
# 6.5 Penyimpanan artefak
# ----------------------------------------------------------------------
section("6.5 Penyimpanan artefak")
np.savez_compressed(f"{ART_DIR}/preprocessed_v1.npz",
    X_train=X_train_sel.values, y_train=y_train.values, resample_idx=resample_idx,
    X_val=X_val_sel.values, y_val=y_val.values, X_test=X_test_sel.values,
    id_train=id_train.values, id_val=id_val.values, id_test=id_test.values,
    feature_names=np.array(X_train_sel.columns))
json.dump({"version": "v1", "label_map": LABEL_MAP, "seed": SEED,
           "n_features": int(X_train_sel.shape[1]),
           "feature_names": list(X_train_sel.columns),
           "outlier_clipping": f"IQR, k={K_FINAL}",
           "encoding_scaling": "one-hot (nominal), standardisasi (numerik dan ordinal)",
           "feature_selection": f"mutual information, n={N_FINAL}",
           "oversampling": "random oversampling pada data train"},
          open(f"{ART_DIR}/preprocessed_meta.json", "w"), indent=2)
print(f"Tersimpan: {ART_DIR.relative_to(ROOT)}/preprocessed_v1.npz dan {ART_DIR.relative_to(ROOT)}/preprocessed_meta.json")
print("Train beroversampling dapat direkonstruksi: X_train[resample_idx], y_train[resample_idx]")


6.1 Perbandingan strategi imbalance (5-fold CV pada train)


                           macroF1     std  F1_Enrolled  Recall_Enrolled
LR  tanpa penanganan        0.6677  0.0131       0.3743           0.3056
LR  class_weight=balanced   0.6995  0.0143       0.5046           0.6360
LR  oversampling            0.6991  0.0164       0.5024           0.6315
MLP tanpa penanganan        0.6727  0.0173       0.3984           0.3640
MLP oversampling            0.6743  0.0172       0.4449           0.5056

Catatan: MLPClassifier tidak mendukung class_weight, sehingga oversampling menjadi satu-satunya cara yang identik untuk versi from scratch dan scikit-learn.

6.2 Oversampling final (hanya data train)
          train_sebelum  train_sesudah  validasi (tidak diubah)
Dropout             795           1236                      199
Enrolled            445           1236                      111
Graduate           1236           1236                      310

Bentuk train sebelum (2476, 20) -> sesudah (3708, 20)

6.3 Verifikasi bebas leakage
[LULUS] ID train dan

## 7. Verifikasi reproduksibilitas
Pemeriksaan bahwa fungsi `prepare_data` pada modul menghasilkan data yang identik dengan hasil notebook.

In [7]:
# @title 7. Verifikasi Reproduksibilitas Pipeline Modul
section("7. Verifikasi reproduksibilitas")

hasil = prepare_data(DATA_DIR)
muat = load_preprocessed(ART_DIR / "preprocessed_v1.npz")

checks = {
    "X_train modul == X_train notebook": np.allclose(hasil.X_train.values, X_train_sel.values),
    "X_val modul == X_val notebook": np.allclose(hasil.X_val.values, X_val_sel.values),
    "X_test modul == X_test notebook": np.allclose(hasil.X_test.values, X_test_sel.values),
    "y_train identik": bool((hasil.y_train.values == y_train.values).all()),
    "Indeks oversampling identik": bool((hasil.resample_idx == resample_idx).all()),
    "Nama fitur identik": hasil.feature_names == list(X_train_sel.columns),
    "Artefak tersimpan == hasil modul (X_train)": np.allclose(muat["X_train"], hasil.X_train.values),
    "Artefak tersimpan: train beroversampling": muat["X_train_res"].shape == (len(resample_idx), hasil.X_train.shape[1]),
}
for k, v in checks.items():
    print(f"[{'LULUS' if v else 'GAGAL'}] {k}")
assert all(checks.values()), "Terdapat ketidakcocokan antara notebook dan modul."
print(f"\nBentuk akhir: train {hasil.X_train.shape}, validasi {hasil.X_val.shape}, test {hasil.X_test.shape}, "
      f"train beroversampling {hasil.X_train_resampled.shape}")


7. Verifikasi reproduksibilitas


[LULUS] X_train modul == X_train notebook
[LULUS] X_val modul == X_val notebook
[LULUS] X_test modul == X_test notebook
[LULUS] y_train identik
[LULUS] Indeks oversampling identik
[LULUS] Nama fitur identik
[LULUS] Artefak tersimpan == hasil modul (X_train)
[LULUS] Artefak tersimpan: train beroversampling

Bentuk akhir: train (2476, 20), validasi (620, 20), test (1328, 20), train beroversampling (3708, 20)
